# Corrupted CSV Data

Real-world CSV data often contains corruption. In this notebook we'll identify and fix various types of data quality issues in a German sales dataset.

By the end of this notebook, you will be able to:
- Load a CSV file and detect hidden data quality issues
- Parse dates with mixed formats using pandas
- Identify and fix rows that use the wrong delimiter
- Build a simple data quality report

## Known Issues in `DE Sales.csv`

The file `DE Sales.csv` contains German sales data with the following known corruption:

| Rows | Issue |
|---|---|
| 96209-96214 | Semicolons (`;`) used instead of commas as delimiter -- the entire row is crammed into a single column |
| 98817-98830 | Dutch date format (`28-7-2004`) instead of US format (`7/28/2004`) |
| 179482-179484 | Truly corrupt garbage data (random strings, not valid records) |
| 232710-232719 | Missing last column (`Country`) |

We will work through these issues step by step.

### Exercise 1: Load the data

Load `DE Sales.csv` using `pd.read_csv()`. Display the first 5 rows using `.head()` and check the column data types using `.dtypes`.

Notice that loading seems to succeed without errors, even though the data has issues. Why might that be?

**Explanation:** We use `pd.read_csv()` to load the CSV file. The loading succeeds because pandas reads everything as strings or infers types column-by-column. The corrupt rows cause the `Date` and `ProductID` columns to be loaded as `object` (string) type instead of their expected types (datetime and integer), but pandas does not raise an error -- it silently falls back to the most general type that can hold all values.

In [ ]:
import pandas as pd

df = pd.read_csv("DE Sales.csv")

print(df.head())
print()
print(df.dtypes)

### Exercise 2: Attempt to parse dates

Try to convert the `Date` column to datetime using `pd.to_datetime(df['Date'], format="mixed")`.

Wrap this in a `try`/`except ValueError` block and print the error message. What does the error tell us about the data?

**Explanation:** We wrap the `pd.to_datetime()` call in a `try`/`except` block because we expect it to fail. The `format="mixed"` parameter tells pandas to try multiple date formats, but it still cannot parse the garbage data in the corrupt rows (e.g. `893qgunheja9;gpuivajnr`). The error message reveals the exact string that caused the failure, which gives us a clue about the nature of the corruption.

In [ ]:
try:
    df['Date'] = pd.to_datetime(df['Date'], format="mixed")
except ValueError as e:
    print(e)

### Exercise 3: Parse dates with error handling

Parse the `Date` column again, but this time pass `errors="coerce"` to `pd.to_datetime()`. This replaces any unparseable dates with `NaT` (Not a Time) instead of raising an error.

After parsing, display rows 179480 through 179486 (using `df[179480:179486]`) to see the corrupt rows. What do you notice about these rows?

**Explanation:** By passing `errors="coerce"`, pandas replaces any value it cannot parse as a date with `NaT` (Not a Time). This allows the conversion to complete without raising an error. When we inspect rows 179480-179486, we can see that the corrupt rows have `NaT` in the Date column and `NaN` in the numeric columns, while the ProductID column contains garbage strings like `VERVUILING` and `qhrn8q2yn`. These rows are clearly not valid sales records.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format="mixed", errors="coerce")

df[179480:179486]

### Exercise 4: Inspect the semicolon rows

Display rows 96209 through 96215 (using `df[96209:96215]`). These rows used semicolons instead of commas as the delimiter when the CSV was created.

Notice how the entire row content has been crammed into the `ProductID` column, and all other columns are `NaN` or `NaT`.

**Explanation:** When we display these rows, we can see that the entire semicolon-separated record (e.g. `631;7/27/2004;20251;1;398.74;Germany`) ended up in the `ProductID` column. This happened because `pd.read_csv()` uses commas as the default delimiter, and since these rows use semicolons, the parser treated the whole line as a single field. The remaining columns are `NaT` or `NaN` because there were no commas to split on.

In [ ]:
df[96209:96215]

### Exercise 5: Fix the semicolon rows

Fix the rows that use semicolons as delimiters:

1. Use `df.query("ProductID.str.contains(';')")` to find all rows where the `ProductID` contains a semicolon.
2. Use `.ProductID.str.split(';', expand=True)` on the result to split the crammed data into separate columns.
3. Assign the correct column names (`["ProductID", "Date", "Zip", "Units", "Revenue", "Country"]`).
4. Use `df.loc[...]` to write the corrected values back into the original DataFrame.

**Explanation:** We first use `df.query()` with `str.contains(';')` to find all rows where the ProductID contains a semicolon. Then we split the ProductID string on semicolons using `str.split(';', expand=True)`, which returns a new DataFrame with one column per split value. We rename the columns to match the original column names, and finally use `df.loc` with the row indices to write the corrected values back into the original DataFrame. This approach preserves the original row indices, so the fixed data ends up in the right place.

In [ ]:
# Find rows with semicolons in ProductID
items_with_wrong_separator = df.query("ProductID.str.contains(';')")

# Split the crammed data into separate columns
items_with_wrong_separator = items_with_wrong_separator.ProductID.str.split(';', expand=True)

# Assign proper column names
column_list = ["ProductID", "Date", "Zip", "Units", "Revenue", "Country"]
items_with_wrong_separator.columns = column_list

# Write corrected values back into the original DataFrame
row_list = items_with_wrong_separator.index.to_list()
df.loc[row_list, column_list] = items_with_wrong_separator

### Exercise 6: Verify the fixes

Display rows 96209-96215 again to confirm the semicolon rows are now fixed.

Also display rows 98817-98830 to verify that pandas correctly handled the Dutch date format (`28-7-2004`) when using `format="mixed"`.

**Explanation:** After the fix, rows 96209-96215 should now show properly separated values in each column. For the Dutch date format rows (98817-98830), pandas' `format="mixed"` option was able to recognize the `28-7-2004` format alongside the US `7/28/2004` format and parse both correctly. This demonstrates that `format="mixed"` is quite flexible, though it can be risky with ambiguous dates (e.g. `1/2/2004` could be January 2 or February 1).

In [ ]:
# Verify the semicolon rows are fixed
print("Rows 96209-96215 (semicolon fix):")
print(df[96209:96215])
print()

# Verify the Dutch date format rows
print("Rows 98817-98830 (Dutch date format):")
print(df[98817:98830])

### Exercise 7: Build a data quality report

Build a simple data quality report by counting:

1. How many rows have `NaT` (missing) dates? Use `df['Date'].isna().sum()`.
2. How many rows have a non-numeric `ProductID`? Use `pd.to_numeric(df['ProductID'], errors='coerce')` and count the resulting `NaN` values.
3. How many rows have a missing `Country`? Use `df['Country'].isna().sum()`.

Print a summary with all three counts.

**Explanation:** We use `.isna().sum()` to count missing values in each column of interest. For the ProductID column, we attempt a numeric conversion with `errors='coerce'` -- any value that cannot be converted to a number becomes `NaN`, and we count those. This gives us a quick overview of how many rows still have data quality issues after our fixes. The missing dates correspond to the corrupt garbage rows, the non-numeric ProductIDs reveal any remaining text in that column, and missing Country values flag rows with incomplete data.

In [ ]:
# Count rows with missing dates
nat_dates = df['Date'].isna().sum()
print(f"Rows with missing (NaT) dates: {nat_dates}")

# Count rows with non-numeric ProductIDs
non_numeric_ids = pd.to_numeric(df['ProductID'], errors='coerce').isna().sum()
print(f"Rows with non-numeric ProductID: {non_numeric_ids}")

# Count rows with missing Country
missing_country = df['Country'].isna().sum()
print(f"Rows with missing Country: {missing_country}")

# Total potentially problematic rows
print(f"\nTotal rows in dataset: {len(df)}")

## Summary

In this notebook we worked through several common types of CSV data corruption:

| Strategy | When to use |
|---|---|
| `format="mixed"` | When dates appear in multiple valid formats |
| `errors="coerce"` | When some values are completely unparseable and should become `NaT` or `NaN` |
| `str.split()` with `expand=True` | When data from multiple columns has been crammed into a single column |
| `df.query()` / boolean indexing | To isolate problematic rows for targeted repair |
| `pd.to_numeric(errors='coerce')` | To detect non-numeric values in columns that should be numeric |

Key takeaways:
- **CSV loading can succeed silently** even when the data is corrupt. Always inspect your data after loading.
- **Fix issues in-place** using `df.loc[...]` to write corrected values back into the DataFrame.
- **Build data quality reports** to quantify the extent of corruption before deciding on a cleaning strategy.